In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gazizovvyacheslav/course1-dataset/sessions_with_session_features.csv
/kaggle/input/datasets/gazizovvyacheslav/course1-dataset/sessions_with_features.csv
/kaggle/input/datasets/gazizovvyacheslav/course1-dataset/sessions_with_session_features_final.csv
/kaggle/input/datasets/gazizovvyacheslav/course1-dataset/sessions_with_session_features_final.parquet
/kaggle/input/datasets/gazizovvyacheslav/course1-dataset/sessions_with_session_features_final/sessions_with_session_features_final.csv
/kaggle/input/datasets/gazizovvyacheslav/user-sessions-with-target/user_sessions_with_target.csv
/kaggle/input/datasets/gazizovvyacheslav/installations/installations-13.csv


# ЧАСТЬ 2

In [6]:
installations_path = "/kaggle/input/datasets/gazizovvyacheslav/installations/installations-13.csv"

installations = pd.read_csv(installations_path)

installations.head()

/tmp/ipykernel_57/2660053637.py:3: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  installations = pd.read_csv(installations_path)


,installation_id,application_id,publisher_name,publisher_id,tracker_name,tracking_id,click_timestamp,click_datetime,click_ipv6,click_url_parameters,...,app_package_name,connection_type,operator_name,mcc,mnc,country_iso_code,city,appmetrica_device_id,attributed_touch_type,windows_aid
0,8f1043b194d64802ab53f658ddbeda3d,4507258,Facebook,5,Autocreated Facebook,749352386736224299,0,1970-01-01 03:00:00,::,utm_source=apps.facebook.com&utm_campaign=fb4a...,...,com.multicastgames.gtarpg,wifi,Maxis/Hotlink,502,12,MY,NaN,8680852242714139433,unknown,NaN
1,0c6da7bd582e4967bfff5e3b4641de5c,4507258,Facebook,5,Autocreated Facebook,749352386736224299,0,1970-01-01 03:00:00,::,utm_source=apps.facebook.com&utm_campaign=fb4a...,...,com.multicastgames.gtarpg,wifi,Orange,231,1,SK,NaN,14178045950997430240,unknown,NaN
2,2f56a4bf235642178312341354a958f4,4507258,Facebook,5,Autocreated Facebook,749352386736224299,0,1970-01-01 03:00:00,::,utm_source=apps.facebook.com&utm_campaign=fb4a...,...,com.multicastgames.gtarpg,wifi,Mobifone,452,1,VN,NaN,17447657440273129823,unknown,NaN
3,bd6286635e234c7ea8c1a28995c82cad,4507258,Unity Ads,72267,Autocreated Unity Ads,461487338288129669,0,1970-01-01 03:00:00,::,adjust_external_click_id=unityads_aa2bac48-af9...,...,com.multicastgames.gtarpg,cell,Orange,260,3,PL,NaN,4954135058209950186,unknown,NaN
4,bfe794486d8240ec91fe56d6ba7ee140,4507258,Unity Ads,72267,Autocreated Unity Ads,461487338288129669,0,1970-01-01 03:00:00,::,adjust_external_click_id=unityads_,...,com.multicastgames.gtarpg,wifi,1O1O / csl / Club Sim,454,0,CN,NaN,14662885543682571931,unknown,NaN


In [7]:
installations.info()
installations.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 879779 entries, 0 to 879778
Data columns (total 43 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   installation_id            879779 non-null  object 
 1   application_id             879779 non-null  int64  
 2   publisher_name             665904 non-null  object 
 3   publisher_id               879779 non-null  int64  
 4   tracker_name               879779 non-null  object 
 5   tracking_id                879779 non-null  int64  
 6   click_timestamp            879779 non-null  int64  
 7   click_datetime             879779 non-null  object 
 8   click_ipv6                 879779 non-null  object 
 9   click_url_parameters       782973 non-null  object 
 10  click_id                   0 non-null       float64
 11  click_user_agent           0 non-null       float64
 12  match_type                 652498 non-null  object 
 13  install_datetime           87

Index(['installation_id', 'application_id', 'publisher_name', 'publisher_id',
       'tracker_name', 'tracking_id', 'click_timestamp', 'click_datetime',
       'click_ipv6', 'click_url_parameters', 'click_id', 'click_user_agent',
       'match_type', 'install_datetime', 'install_timestamp',
       'install_receive_datetime', 'install_receive_timestamp', 'install_ipv6',
       'is_reinstallation', 'is_reattribution', 'ios_ifa', 'ios_ifv',
       'android_id', 'google_aid', 'profile_id', 'os_name', 'os_version',
       'oaid', 'device_manufacturer', 'device_model', 'device_type',
       'device_locale', 'app_version_name', 'app_package_name',
       'connection_type', 'operator_name', 'mcc', 'mnc', 'country_iso_code',
       'city', 'appmetrica_device_id', 'attributed_touch_type', 'windows_aid'],
      dtype='object')

Возьмем такие признаки:

time_since_install_sec - время с установки

traffic_source - источник трафика

tracker_name - рекламная кампания

is_reinstallation - повторная установка

is_reattribution - повторная атрибуция

attributed_touch_type - тип атрибуции

install_hour - час установки

install_dayofweek - день установки

install_country - страна установки

install_device_type - тип устройства

install_app_version_name - версия приложения

install_connection_type - тип подключения

In [10]:
sessions_path = "/kaggle/input/datasets/gazizovvyacheslav/course1-dataset/sessions_with_session_features_final.csv"

sessions = pd.read_csv(sessions_path)

install_cols = [
    "installation_id",
    "install_datetime",
    "publisher_name",
    "tracker_name",
    "is_reinstallation",
    "is_reattribution",
    "attributed_touch_type",
    "country_iso_code",
    "device_type",
    "app_version_name",
    "connection_type"
]

installations_small = installations[install_cols].drop_duplicates("installation_id").copy()

installations_small["install_datetime"] = pd.to_datetime(
    installations_small["install_datetime"],
    errors="coerce"
)

sessions["start"] = pd.to_datetime(
    sessions["start"],
    errors="coerce"
)

sessions = sessions.merge(
    installations_small,
    on="installation_id",
    how="left"
)

sessions["time_since_install_sec"] = (
    sessions["start"] - sessions["install_datetime"]
).dt.total_seconds()

sessions["traffic_source"] = sessions["publisher_name"].fillna("unknown")

sessions["install_hour"] = sessions["install_datetime"].dt.hour
sessions["install_dayofweek"] = sessions["install_datetime"].dt.dayofweek

sessions["install_country"] = sessions["country_iso_code"]
sessions["install_device_type"] = sessions["device_type"]
sessions["install_app_version_name"] = sessions["app_version_name"]
sessions["install_connection_type"] = sessions["connection_type"]

install_feature_cols = [
    "time_since_install_sec",
    "traffic_source",
    "tracker_name",
    "is_reinstallation",
    "is_reattribution",
    "attributed_touch_type",
    "install_hour",
    "install_dayofweek",
    "install_country",
    "install_device_type",
    "install_app_version_name",
    "install_connection_type"
]

display(sessions[install_feature_cols].head())

,time_since_install_sec,traffic_source,tracker_name,is_reinstallation,is_reattribution,attributed_touch_type,install_hour,install_dayofweek,install_country,install_device_type,install_app_version_name,install_connection_type
0,73366.0,Google Ads,Autocreated Google Ads,False,False,unknown,2.0,4.0,EC,phone,0.8.7,wifi
1,90990.0,Google Ads,Autocreated Google Ads,False,False,unknown,21.0,3.0,ZA,phone,0.8.7,wifi
2,87573.0,AppLovin,Autocreated AppLovin,False,False,unknown,22.0,3.0,PL,phone,0.8.7,wifi
3,1026262.0,AppLovin,Autocreated AppLovin,True,True,unknown,1.0,0.0,PL,phone,0.8.7,wifi
4,37969.0,unknown,Google Play,False,False,unknown,12.0,4.0,RU,phone,0.9.2,cell


install_country             0.028085
time_since_install_sec      0.019059
is_reinstallation           0.019059
tracker_name                0.019059
install_dayofweek           0.019059
is_reattribution            0.019059
install_hour                0.019059
attributed_touch_type       0.019059
install_app_version_name    0.019059
install_device_type         0.019059
install_connection_type     0.019059
traffic_source              0.000000
dtype: float64

## EDA

Теперь посмотрим что за данные

In [11]:
sessions.shape

(4153829, 27)

ПРопуски


In [18]:
eda_cols = install_feature_cols.copy()

display(sessions[eda_cols].head())

display(sessions[eda_cols].describe(include="all").T)

display(sessions[eda_cols].isna().mean().sort_values(ascending=False))

,time_since_install_sec,traffic_source,tracker_name,is_reinstallation,is_reattribution,attributed_touch_type,install_hour,install_dayofweek,install_country,install_device_type,install_app_version_name,install_connection_type
0,73366.0,Google Ads,Autocreated Google Ads,False,False,unknown,2.0,4.0,EC,phone,0.8.7,wifi
1,90990.0,Google Ads,Autocreated Google Ads,False,False,unknown,21.0,3.0,ZA,phone,0.8.7,wifi
2,87573.0,AppLovin,Autocreated AppLovin,False,False,unknown,22.0,3.0,PL,phone,0.8.7,wifi
3,1026262.0,AppLovin,Autocreated AppLovin,True,True,unknown,1.0,0.0,PL,phone,0.8.7,wifi
4,37969.0,unknown,Google Play,False,False,unknown,12.0,4.0,RU,phone,0.9.2,cell


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
time_since_install_sec,4074661.0,NaN,NaN,NaN,1197640.408535,2816073.938861,-3852147.0,12275.0,229107.0,1039124.0,31451791.0
traffic_source,4153829,9,Google Ads,1999092,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tracker_name,4074661,11,Autocreated Google Ads,1999092,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_reinstallation,4074661,2,False,3722795,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_reattribution,4074661,2,False,3722795,NaN,NaN,NaN,NaN,NaN,NaN,NaN
attributed_touch_type,4074661,1,unknown,4074661,NaN,NaN,NaN,NaN,NaN,NaN,NaN
install_hour,4074661.0,NaN,NaN,NaN,12.319441,7.116937,0.0,6.0,13.0,19.0,23.0
install_dayofweek,4074661.0,NaN,NaN,NaN,3.008685,2.035672,0.0,1.0,3.0,5.0,6.0
install_country,4037168,207,US,479173,NaN,NaN,NaN,NaN,NaN,NaN,NaN
install_device_type,4074661,2,phone,3805557,NaN,NaN,NaN,NaN,NaN,NaN,NaN


install_country             0.028085
time_since_install_sec      0.019059
is_reinstallation           0.019059
tracker_name                0.019059
install_dayofweek           0.019059
is_reattribution            0.019059
install_hour                0.019059
attributed_touch_type       0.019059
install_app_version_name    0.019059
install_device_type         0.019059
install_connection_type     0.019059
traffic_source              0.000000
dtype: float64

Большая часть сессий успешно имеет всю информацию об установке пользователя. Большиснвто пользоватедей пришли из Google Ads, с устройствами типа phone, подключением wifi и версией приложения 0.9.7. Признак attributed_touch_type оказался вообще бесполезен. так как принимает только одно значение. Также обнаружены пропуски после объединения таблиц и аномалии во временном признаке time_since_install_sec, где часть сессий вообще начинается раньше времени установки.

Разберемся что же за аномалии в признаке time_since_install_sec

In [22]:
negative_time = sessions[sessions["time_since_install_sec"] < 0]

display(negative_time.shape)
negative_time[["installation_id", "start", "install_datetime", "time_since_install_sec"]].head(15)


(810880, 28)

,installation_id,start,install_datetime,time_since_install_sec
19,56ce5634936140298f789871864afd56,2025-10-31 23:00:00,2025-10-31 23:56:19,-3379.0
24,a45708c7e18e43e9a0688c73c1d79ed7,2025-10-31 23:00:01,2025-10-31 23:38:29,-2308.0
40,841b02da606949e4b77f14090f8f2ed8,2025-10-31 23:00:02,2025-10-31 23:48:40,-2918.0
70,64c8d036703d4dadb9423690bb9f94f5,2025-10-31 23:00:05,2025-10-31 23:00:45,-40.0
86,028a14f5867a4e548f0b804f77a30f91,2025-10-31 23:00:07,2025-10-31 23:27:11,-1624.0
107,0551130a838244f8bad2774ef50128fa,2025-10-31 23:00:09,2025-10-31 23:19:18,-1149.0
146,e51ab0657c3944549fc95e8ca9e4fac4,2025-10-31 23:00:18,2025-10-31 23:10:16,-598.0
154,8bdc117e9f884b3e9ada3d2387a3be06,2025-10-31 23:00:20,2025-10-31 23:39:28,-2348.0
163,92634a8910284dfab6ab2531a2c71773,2025-10-31 23:00:22,2025-10-31 23:27:13,-1611.0
185,223a270addc64528b84834cb9a683aff,2025-10-31 23:00:31,2025-10-31 23:47:08,-2797.0


In [21]:
(sessions["time_since_install_sec"] < 0).mean()


np.float64(0.19521265800782844)

Целых 20 процентов данных, что то не то с ними, ошибка не просто в выбросах, а системная

In [23]:
neg_stats = (
    negative_time["time_since_install_sec"]
    .abs()
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
)

display(neg_stats)

count    8.108800e+05
mean     7.479011e+03
std      3.894067e+04
min      1.000000e+00
25%      3.203000e+03
50%      3.597000e+03
75%      3.598000e+03
90%      3.598000e+03
95%      3.598000e+03
99%      1.701191e+05
max      3.852147e+06
Name: time_since_install_sec, dtype: float64

у большинства отрицательных строк сессия раньше установки примерно на один час. Это похоже не на реальные ошибки поведения, а на техническую проблему: start записан как начало часового интервала, а install_datetime записан точным временем внутри этого часа

То есть проблема вероятно в округлении



In [31]:
sessions.loc[
    (sessions["time_since_install_sec"] < 0) &
    (sessions["time_since_install_sec"] >= -3600),
    "time_since_install_sec"
] = 0

sessions.loc[
    sessions["time_since_install_sec"] < -3600,
    "time_since_install_sec"
] = np.nan

In [32]:
(sessions["time_since_install_sec"] < 0).mean()

np.float64(0.0)

Решили проблему

In [37]:
target_col = "target_next_session_length_sec"
num_cols = [
    "time_since_install_sec",
    "install_hour",
    "install_dayofweek"
]

corr_df = sessions[num_cols + [target_col]].corr(method="pearson")

display(corr_df)

,time_since_install_sec,install_hour,install_dayofweek,target_next_session_length_sec
time_since_install_sec,1.000000,-0.012354,0.009368,-0.003845
install_hour,-0.012354,1.000000,0.025407,0.000889
install_dayofweek,0.009368,0.025407,1.000000,-0.000756
target_next_session_length_sec,-0.003845,0.000889,-0.000756,1.000000


Числовые install признаки почти не имеют линейной зависимости с target_next_session_length_sec (таргетом), но ниче страшного. для нелинейных алгоритмов они могут пригодиться.

Посмотрим топ значений у категориальных признаков


In [38]:
cat_cols = [
    "traffic_source",
    "tracker_name",
    "is_reinstallation",
    "is_reattribution",
    "attributed_touch_type",
    "install_country",
    "install_device_type",
    "install_app_version_name",
    "install_connection_type"
]

for i in cat_cols:
    print("\n")
    print(i)
    display(sessions[i].value_counts(dropna=False).head(15))



 traffic_source


traffic_source
Google Ads                  1999092
unknown                      976455
AppLovin                     880629
Unity Ads                    275444
Facebook                      17415
Mintegral                      4677
TikTok Ads                      104
SHAREit                          11
Unknown (source: Adjust)          2
Name: count, dtype: int64


 tracker_name


tracker_name
Autocreated Google Ads                  1999092
Autocreated AppLovin                     880629
unknown                                  461717
Google Play                              434563
Autocreated Unity Ads                    275444
NaN                                       79168
Autocreated Facebook                      17415
Autocreated Mintegral                      4677
Google Search                              1007
Autocreated TikTok Ads                      104
Autocreated SHAREit                          11
Autocreated Unknown (source: Adjust)          2
Name: count, dtype: int64


 is_reinstallation


...

install_device_type
phone     3805557
tablet     269104
NaN         79168
Name: count, dtype: int64


 install_app_version_name


install_app_version_name
0.9.7     1847366
0.9.6      977321
0.9.9      444336
0.9.4      366694
0.8.7      293142
NaN         79168
1.0.0       36020
0.6.11      24384
0.8.5       16918
0.8.6       12169
0.8.1       11325
0.9.2       10960
0.7.5        5798
0.5.1        5428
0.7.9        4436
Name: count, dtype: int64


 install_connection_type


install_connection_type
wifi       2525524
cell       1542145
NaN          79168
unknown       6992
Name: count, dtype: int64

Исключаем attributed_touch_type


In [41]:
install_feature_cols = [
    "time_since_install_sec",
    "traffic_source",
    "tracker_name",
    "is_reinstallation",
    "is_reattribution",
    "install_hour",
    "install_dayofweek",
    "install_country",
    "install_device_type",
    "install_app_version_name",
    "install_connection_type"
]

cat_cols = [
    "traffic_source",
    "tracker_name",
    "is_reinstallation",
    "is_reattribution",
    "install_country",
    "install_device_type",
    "install_app_version_name",
    "install_connection_type"
]


Создадим признак на органику

In [45]:
organic_sources = ["Google Play", "Google Search"]
sessions["is_organic"] = sessions["tracker_name"].isin(organic_sources).astype(int)
install_feature_cols.append("is_organic")

In [46]:
sessions["is_organic"].value_counts(dropna=False)

is_organic
0    3718259
1     435570
Name: count, dtype: int64

In [48]:
feature_target_spread = {}

for i in cat_cols + ["is_organic"]:
    stat = (
        sessions
        .groupby(i, dropna=False)[target_col]
        .agg(["count", "mean"])
    )
    
    stat = stat[stat["count"] >= 1000]
    
    if len(stat) > 1:
        feature_target_spread[i] = stat["mean"].max() - stat["mean"].min()

pd.Series(feature_target_spread).sort_values(ascending=False)

install_app_version_name    6363.497612
install_country             5260.733315
install_connection_type     1068.390240
tracker_name                 827.599043
install_device_type          672.748016
traffic_source               579.971087
is_organic                   469.861987
is_reattribution             378.038120
is_reinstallation            378.038120
dtype: float64

In [49]:
cat_summary = pd.DataFrame({
    "n_unique": sessions[cat_cols].nunique(dropna=True),
    "missing_rate": sessions[cat_cols].isna().mean(),
    "top_value": [sessions[i].value_counts(dropna=True).index[0] for i in cat_cols],
    "top_value_share": [sessions[i].value_counts(dropna=True, normalize=True).iloc[0] for i in cat_cols]
}).sort_values("n_unique", ascending=False)

display(cat_summary)

,n_unique,missing_rate,top_value,top_value_share
install_country,207,0.028085,US,0.118690
install_app_version_name,48,0.019059,0.9.7,0.453379
tracker_name,11,0.019059,Autocreated Google Ads,0.490616
traffic_source,9,0.000000,Google Ads,0.481265
install_connection_type,3,0.019059,wifi,0.619812
is_reinstallation,2,0.019059,False,0.913645
is_reattribution,2,0.019059,False,0.913645
install_device_type,2,0.019059,phone,0.933957


<span style="color:red">Как видно из таблицы все категориальные признаки кроме первых двух можно кодировать one-hot</span>

В install_app_version_name видим перекос в 0.9.7, значит можно редкие категории закодировать в other, а топовые оставить.
В install_country тоже есть перекос, думаю аналогично сгруппировать редкие страны в other, потом one-hot

Для CatBoost эти признаки могут использоваться напрямую после заполнения пропусков. То есть пока не будем кодировать категориальные.

ПРопуски в категориальных признаках заменим на unknown

In [53]:
cat_cols_for_model = [
    "traffic_source",
    "tracker_name",
    "install_country",
    "install_device_type",
    "install_app_version_name",
    "install_connection_type"
]

for i in cat_cols_for_model:
    sessions[i] = sessions[i].fillna("unknown").astype(str)

boolean пропуски просто заменим на False

In [54]:
bool_cols = ["is_reinstallation", "is_reattribution"]

for i in bool_cols:
    sessions[i] = (
        sessions[col]
        .fillna(False)
        .astype(str)
        .str.lower()
        .map({"true": 1, "false": 0})
        .fillna(0)
        .astype(int)
    )

Ну и переведем boolean признаки в нолики и единички

In [55]:
bool_cols = ["is_reinstallation", "is_reattribution"]

for i in bool_cols:
    sessions[i] = (
        sessions[i]
        .astype(str)
        .str.lower()
        .map({"true": 1, "false": 0})
        .fillna(0)
        .astype(int)
    )

Числовые же пропуски не удалим, а заполним отдельным значением и добавим флаг пропуска

In [56]:
num_cols_for_model = [
    "time_since_install_sec",
    "install_hour",
    "install_dayofweek"
]

for i in num_cols_for_model:
    sessions[i+"_is_missing"] = sessions[i].isna().astype(int)
    sessions[i] = sessions[i].fillna(-1)

In [59]:
sessions[install_feature_cols].isna().mean().sort_values(ascending=False)

time_since_install_sec               0.0
traffic_source                       0.0
tracker_name                         0.0
is_reinstallation                    0.0
is_reattribution                     0.0
install_hour                         0.0
install_dayofweek                    0.0
install_country                      0.0
install_device_type                  0.0
install_app_version_name             0.0
install_connection_type              0.0
is_organic                           0.0
time_since_install_sec_is_missing    0.0
install_hour_is_missing              0.0
install_dayofweek_is_missing         0.0
dtype: float64

Как видно все пропуски почистили

In [58]:
install_feature_cols = [
    "time_since_install_sec",
    "traffic_source",
    "tracker_name",
    "is_reinstallation",
    "is_reattribution",
    "install_hour",
    "install_dayofweek",
    "install_country",
    "install_device_type",
    "install_app_version_name",
    "install_connection_type",
    "is_organic",
    "time_since_install_sec_is_missing",
    "install_hour_is_missing",
    "install_dayofweek_is_missing"
]

Сохраняем

In [60]:
sessions.to_csv("sessions_with_session_and_install_features.csv", index=False)
sessions.to_parquet("/kaggle/working/sessions_with_session_and_install_features.parquet", index=False)

In [61]:
sessions.head()

,appmetrica_device_id,installation_id,session_id,start,end,duration_seconds,duration_hms,events_count,target_next_session_length_sec,install_datetime,...,install_country,install_device_type,install_app_version_name,install_connection_type,time_since_install_days,time_since_install_sec_fixed,is_organic,time_since_install_sec_is_missing,install_hour_is_missing,install_dayofweek_is_missing
0,1219361434347362249,d7222de1105c40159efdf16a07bc2080,10000000013,2025-10-31 23:00:00,2025-10-31 23:13:56,836,00:13:56,200,222.0,2025-10-31 02:37:14,...,EC,phone,0.8.7,wifi,0.849144,73366.0,0,0,0,0
1,12768366038869339700,1a284947f3484798a3d3548689fa35d3,10000000085,2025-10-31 23:00:00,2025-10-31 23:01:39,99,00:01:39,15,32.0,2025-10-30 21:43:30,...,ZA,phone,0.8.7,wifi,1.053125,90990.0,0,0,0,0
2,138580996056989852,d5828d05106a4a51839c585dd22216e3,10000000015,2025-10-31 23:00:00,2025-10-31 23:37:48,2268,00:37:48,280,NaN,2025-10-30 22:40:27,...,PL,phone,0.8.7,wifi,1.013576,87573.0,0,0,0,0
3,14040710517756309412,46399f85c03147e4a122b052b97d7bb8,10000000088,2025-10-31 23:00:00,2025-10-31 23:01:58,118,00:01:58,20,1983.0,2025-10-20 01:55:38,...,PL,phone,0.8.7,wifi,11.878032,1026262.0,0,0,0,0
4,14307331389053927226,9bb8dd8d945e4b6f9e45ac34f2756429,10000000006,2025-10-31 23:00:00,2025-10-31 23:05:18,318,00:05:18,20,340.0,2025-10-31 12:27:11,...,RU,phone,0.9.2,cell,0.439456,37969.0,1,0,0,0
